In [15]:
import pandas as pd
import numpy as np

In [16]:
# 1. Import and clean column names
df=pd.read_csv("RealEstate_Prices.csv")
df.columns = (df.columns.str.strip().str.lower().str.replace(r"[^0-9a-zA-Z]+", "_", regex=True).str.strip("_"))
print(df.columns)

Index(['property_id', 'neighborhood', 'property_type', 'year_sold', 'bedrooms',
       'bathrooms', 'area_sqft', 'age_of_property_yrs', 'parking_spaces',
       'furnishing_status', 'price'],
      dtype='object')


In [17]:
# 2. Missing values
print(df.isnull().sum())

for c in df.select_dtypes("number"):
    df[c] = df[c].fillna(df[c].median())

for c in df.select_dtypes("object"):
    df[c] = df[c].fillna(df[c].mode()[0]) 


property_id             0
neighborhood           17
property_type          25
year_sold               0
bedrooms               41
bathrooms              33
area_sqft              40
age_of_property_yrs    33
parking_spaces          0
furnishing_status      39
price                  16
dtype: int64


In [18]:
# 3. Merging with an additional dataset (neighborhood information)
neigh = pd.read_csv("Neighborhood_Info.csv")
df = df.merge(neigh, on="neighborhood", how="left")

In [19]:
# 4. Filtering and subsetting
subset = df[(df["property_type"] == "Apartment") & (df["year_sold"] >= 2015) & (df["price"] > 0)]

print(subset.shape) 


(454, 16)


In [20]:
# 5. Encoding categorical variables
df["property_type_label"] = df["property_type"].astype("category").cat.codes
df_encoded = pd.get_dummies(df, columns=["neighborhood"], drop_first=True)

In [21]:
# 6. Aggregation
print(df.groupby("neighborhood")["price"].mean().sort_values(ascending=False))
print(df.groupby("property_type")["price"].agg(["count", "mean", "median", "max"])) 

neighborhood
Koregaon Park     3.685866e+07
Kothrud           3.084591e+07
Viman Nagar       2.794003e+07
Kharadi           2.725685e+07
Baner             2.621731e+07
Wakad             2.182028e+07
Aundh             2.097311e+07
Hadapsar          1.961804e+07
Hinjewadi         1.795111e+07
Pimple Saurabh    1.691015e+07
Name: price, dtype: float64
               count          mean      median          max
property_type                                              
Apartment        697  1.681827e+07  14773000.0  176310000.0
Penthouse         96  6.209034e+07  58740500.0  129836000.0
Row House        153  3.180054e+07  28513000.0   77536000.0
Studio           152  5.321707e+06   4921000.0   18878000.0
Villa            102  6.396757e+07  61208000.0  314494000.0


In [22]:
# 7. Outliers (IQR method)
q1, q3 = df["price"].quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr

print("Outliers found:", ((df["price"] < low) | (df["price"] > high)).sum())
df = df[(df["price"] >= low) & (df["price"] <= high)]
print("Shape after removing outliers:", df.shape)


Outliers found: 115
Shape after removing outliers: (1085, 17)
